In [1]:
%%writefile prompt_library.py
"""
prompt_library.py
==================
A small library of reusable, documented prompt templates for common LLM
tasks. Every template instructs the model to respond with STRICT JSON
ONLY (no markdown fences, no commentary), so the output can be safely
parsed with `json.loads()` downstream.
"""

# 1. SUMMARIZE
# Placeholders: {text}
# Expected JSON: {"summary": "...", "key_points": ["...", "..."]}
SUMMARIZE_PROMPT = """You are a precise summarization engine.

Summarize the following text in 1-2 sentences, and list up to 5 key points.

Text:
\"\"\"{text}\"\"\"

Respond with STRICT JSON ONLY, matching this exact schema, and nothing else
(no markdown fences, no explanations):
{{
  "summary": "<string>",
  "key_points": ["<string>", "..."]
}}"""


# 2. CLASSIFY
# Placeholders: {text}, {categories}
# Expected JSON: {"category": "...", "confidence": 0.0-1.0, "reasoning": "..."}
CLASSIFY_PROMPT = """You are a strict text classifier.

Classify the following text into exactly ONE of these categories:
{categories}

Text:
\"\"\"{text}\"\"\"

Respond with STRICT JSON ONLY, matching this exact schema, and nothing else
(no markdown fences, no explanations):
{{
  "category": "<string, must be one of the given categories>",
  "confidence": <number between 0 and 1>,
  "reasoning": "<short string>"
}}"""


# 3. EXTRACT
# Placeholders: {text}, {fields}
# Expected JSON: {"extracted": {...}, "missing_fields": [...]}
EXTRACT_PROMPT = """You are an information-extraction engine.

From the text below, extract these fields: {fields}
If a field is not present in the text, set its value to null and also
list it under "missing_fields".

Text:
\"\"\"{text}\"\"\"

Respond with STRICT JSON ONLY, matching this exact schema, and nothing else
(no markdown fences, no explanations):
{{
  "extracted": {{"<field_name>": "<value or null>"}},
  "missing_fields": ["<string>", "..."]
}}"""


# 4. SENTIMENT
# Placeholders: {text}
# Expected JSON: {"sentiment": "...", "score": -1.0 to 1.0, "emotions": [...]}
SENTIMENT_PROMPT = """You are a sentiment analysis engine.

Analyze the sentiment of the following text.

Text:
\"\"\"{text}\"\"\"

Respond with STRICT JSON ONLY, matching this exact schema, and nothing else
(no markdown fences, no explanations):
{{
  "sentiment": "positive" | "negative" | "neutral",
  "score": <number between -1 and 1>,
  "emotions": ["<string>", "..."]
}}"""


PROMPT_LIBRARY = {
    "summarize": SUMMARIZE_PROMPT,
    "classify": CLASSIFY_PROMPT,
    "extract": EXTRACT_PROMPT,
    "sentiment": SENTIMENT_PROMPT,
}

Writing prompt_library.py


In [2]:
%%writefile robust_json_prompt.py
"""
robust_json_prompt.py
======================
Asks a model to respond in JSON, parses with json.loads inside a
try/except, retries on failure, and never crashes the program.
"""

import json
import os
import random

from prompt_library import SUMMARIZE_PROMPT


def call_model_real(prompt: str) -> str:
    import anthropic
    client = anthropic.Anthropic()
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


def call_model_mock(prompt: str, force_bad: bool = False) -> str:
    bad_responses = [
        'Sure! Here is the JSON you asked for: {"summary": "Looks ok but has a preamble"}',
        '{"summary": "truncated json", "key_points": ["a", "b"',
        "I cannot help with that request.",
        '{"summary": "trailing comma issue", "key_points": ["a", "b",]}',
    ]
    good_response = json.dumps({
        "summary": "This is a clean, valid summary.",
        "key_points": ["point one", "point two", "point three"],
    })
    if force_bad:
        return random.choice(bad_responses)
    return random.choice([good_response] + bad_responses)


def call_model(prompt: str, force_bad: bool = False) -> str:
    if os.environ.get("ANTHROPIC_API_KEY"):
        return call_model_real(prompt)
    return call_model_mock(prompt, force_bad=force_bad)


def get_json_response(prompt: str, max_retries: int = 2, force_bad: bool = False):
    attempt = 0
    last_error = None
    last_raw = None

    while attempt <= max_retries:
        attempt += 1
        try:
            raw = call_model(prompt, force_bad=force_bad)
            last_raw = raw

            cleaned = raw.strip()
            if cleaned.startswith("```"):
                cleaned = cleaned.strip("`")
                if cleaned.lower().startswith("json"):
                    cleaned = cleaned[4:]
                cleaned = cleaned.strip()

            parsed = json.loads(cleaned)
            return parsed

        except json.JSONDecodeError as e:
            last_error = e
            print(f"  [attempt {attempt}/{max_retries + 1}] JSON parse failed: {e}")
        except Exception as e:
            last_error = e
            print(f"  [attempt {attempt}/{max_retries + 1}] Unexpected error: {e}")

    print(
        f"  ERROR: could not get valid JSON after {max_retries + 1} attempts. "
        f"Last error: {last_error}\n"
        f"  Last raw response was: {last_raw!r}"
    )
    return None


def run_test_suite():
    test_cases = [
        {"name": "Case 1 - normal text, allow random mock behavior", "force_bad": False,
         "text": "The quarterly results showed strong growth in all regions."},
        {"name": "Case 2 - force a bad/malformed response", "force_bad": True,
         "text": "Some text that will get a broken JSON reply."},
        {"name": "Case 3 - normal text, allow random mock behavior", "force_bad": False,
         "text": "Customer support tickets increased 12% this month."},
        {"name": "Case 4 - force a bad/malformed response", "force_bad": True,
         "text": "Another input guaranteed to trigger bad JSON."},
        {"name": "Case 5 - empty input text", "force_bad": False, "text": ""},
    ]

    results = []
    for i, case in enumerate(test_cases, start=1):
        print(f"\n=== {case['name']} ===")
        prompt = SUMMARIZE_PROMPT.format(text=case["text"])
        try:
            result = get_json_response(prompt, max_retries=2, force_bad=case["force_bad"])
            if result is not None:
                print(f"  SUCCESS -> {result}")
                results.append((i, "parsed", result))
            else:
                print("  Handled gracefully -> no crash, error was reported above.")
                results.append((i, "failed_gracefully", None))
        except Exception as e:
            print(f"  UNEXPECTED CRASH PREVENTED: {e}")
            results.append((i, "unexpected_exception_caught", str(e)))

    print("\n=== Test Summary ===")
    for i, status, _ in results:
        print(f"  Test {i}: {status}")
    print(f"\nProgram completed all {len(test_cases)} tests without crashing.")
    return results


if __name__ == "__main__":
    run_test_suite()

Writing robust_json_prompt.py


In [3]:
!python3 robust_json_prompt.py


=== Case 1 - normal text, allow random mock behavior ===
  [attempt 1/3] JSON parse failed: Expecting ',' delimiter: line 1 column 54 (char 53)
  [attempt 2/3] JSON parse failed: Expecting value: line 1 column 1 (char 0)
  [attempt 3/3] JSON parse failed: Expecting value: line 1 column 1 (char 0)
  ERROR: could not get valid JSON after 3 attempts. Last error: Expecting value: line 1 column 1 (char 0)
  Last raw response was: 'I cannot help with that request.'
  Handled gracefully -> no crash, error was reported above.

=== Case 2 - force a bad/malformed response ===
  [attempt 1/3] JSON parse failed: Expecting value: line 1 column 1 (char 0)
  [attempt 2/3] JSON parse failed: Expecting ',' delimiter: line 1 column 54 (char 53)
  [attempt 3/3] JSON parse failed: Expecting ',' delimiter: line 1 column 54 (char 53)
  ERROR: could not get valid JSON after 3 attempts. Last error: Expecting ',' delimiter: line 1 column 54 (char 53)
  Last raw response was: '{"summary": "truncated json", "ke